# Import and Shared functions

In [1]:
import os, sys, re, ast
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

AXES = ["M1", "M2", "M3"]   # Base-frame motor axes -- shared by every θ/ω field in both KFLOG and PIDLOG

def parse_val(v):
    v = v.strip()
    try:
        return float(v)
    except ValueError:
        return v

def parse_payload_line(line, tag):
    """Split 'TIMESTAMP INFO <TAG> {dict}' and literal_eval the trailing dict."""
    if f" {tag} " not in line:
        return None
    ts, _, body = line.partition(f" {tag} ")
    ts = ts.split(" INFO")[0].strip()
    try:
        payload = ast.literal_eval(body.strip())
    except (ValueError, SyntaxError):
        return None
    rec = {"timestamp": ts}
    for key, val in payload.items():
        if isinstance(val, list):
            for i, v in enumerate(val):
                rec[f"{key}_{i+1}"] = parse_val(str(v))
        else:
            rec[key] = parse_val(str(val))
    return rec

def load_kf_pid(log_path):
    """Parse KFLOG and PIDLOG lines from a driver log (Config.log_position = true) into two DataFrames."""
    if not os.path.exists(log_path):
        raise FileNotFoundError(f"log_path does not exist: {log_path!r}")
    kf_rows, pid_rows = [], []
    # encoding='utf-8' is required: the driver writes θ/ω/Δ/α as UTF-8 (log.py's
    # RotatingFileHandler is pinned to utf-8), but open() without an explicit encoding
    # falls back to the OS default -- cp1252 on Windows -- which silently mangles those
    # keys instead of raising, so columns like "θ_sp_1" quietly never get created.
    with open(log_path, encoding="utf-8") as f:
        for line in f:
            if " KFLOG " in line:
                rec = parse_payload_line(line, "KFLOG")
                if rec: kf_rows.append(rec)
            elif " PIDLOG " in line:
                rec = parse_payload_line(line, "PIDLOG")
                if rec: pid_rows.append(rec)

    if not kf_rows and not pid_rows:
        raise ValueError(f"No KFLOG/PIDLOG lines found in {log_path!r} -- was Config.log_position true during this session?")

    def finalize(rows):
        df = pd.DataFrame(rows)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.sort_values("timestamp").reset_index(drop=True)
        df["t_sec"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
        return df

    kf_df  = finalize(kf_rows)  if kf_rows  else pd.DataFrame()
    pid_df = finalize(pid_rows) if pid_rows else pd.DataFrame()
    return kf_df, pid_df


# Load data

In [2]:
# ── Choose log path (last set log_path is what is used) ────────────────────────
log_path = '../logs/alpaca.log'

kf_df, pid_df = load_kf_pid(log_path)
print(f"KF  samples: {len(kf_df)}" + (f"  ({kf_df.t_sec.iloc[-1]:.1f}s span)" if len(kf_df) else ""))
print(f"PID samples: {len(pid_df)}" + (f"  ({pid_df.t_sec.iloc[-1]:.1f}s span)" if len(pid_df) else ""))
print()
print("KF columns:", list(kf_df.columns))
print("PID columns:", list(pid_df.columns))


KF  samples: 15  (2.8s span)
PID samples: 15  (2.8s span)

KF columns: ['timestamp', 'θ_meas_1', 'θ_meas_2', 'θ_meas_3', 'θ_state_1', 'θ_state_2', 'θ_state_3', 'K_gain_1', 'K_gain_2', 'K_gain_3', 'K_gain_4', 'K_gain_5', 'K_gain_6', 'ω_meas_1', 'ω_meas_2', 'ω_meas_3', 'ω_state_1', 'ω_state_2', 'ω_state_3', 'ω_ref_1', 'ω_ref_2', 'ω_ref_3', 't_sec']
PID columns: ['timestamp', 'Δ_sp_1', 'Δ_sp_2', 'Δ_sp_3', 'Δ_pv_1', 'Δ_pv_2', 'Δ_pv_3', 'α_sp_1', 'α_sp_2', 'α_sp_3', 'α_pv_1', 'α_pv_2', 'α_pv_3', 'θ_sp_1', 'θ_sp_2', 'θ_sp_3', 'θ_pv_1', 'θ_pv_2', 'θ_pv_3', 'ω_kp_1', 'ω_kp_2', 'ω_kp_3', 'ω_ki_1', 'ω_ki_2', 'ω_ki_3', 'ω_kd_1', 'ω_kd_2', 'ω_kd_3', 'ω_ff_1', 'ω_ff_2', 'ω_ff_3', 'ω_op_1', 'ω_op_2', 'ω_op_3', 't_sec']


# Combined KF+PID timeline

Merges the two streams on nearest timestamp (they tick independently -- KF on every 518
message, PID on every control step -- so this aligns them within a short tolerance rather
than assuming a shared clock).

In [3]:
TOLERANCE = pd.Timedelta("100ms")

merged = pd.DataFrame()
if len(kf_df) and len(pid_df):
    merged = pd.merge_asof(
        pid_df.sort_values("timestamp"), kf_df.sort_values("timestamp"),
        on="timestamp", direction="nearest", tolerance=TOLERANCE,
        suffixes=("_pid", "_kf"),
    )
    merged["t_sec"] = (merged["timestamp"] - merged["timestamp"].iloc[0]).dt.total_seconds()
print(f"Merged samples: {len(merged)}")
merged.head()


Merged samples: 15


,timestamp,Δ_sp_1,Δ_sp_2,Δ_sp_3,Δ_pv_1,Δ_pv_2,Δ_pv_3,α_sp_1,α_sp_2,α_sp_3,...,ω_meas_2,ω_meas_3,ω_state_1,ω_state_2,ω_state_3,ω_ref_1,ω_ref_2,ω_ref_3,t_sec_kf,t_sec
0,2026-08-25 11:34:55.452,235.874385,-43.976989,77.515954,293.098100,-67.954829,8.078397,238.456864,42.014373,-22.108384,...,-0.000036,-0.000281,-0.000003,-0.000033,-0.000058,0.0,0.0,0.0,0.000,0.000
1,2026-08-25 11:34:55.652,235.874385,-43.976989,77.515954,293.099086,-67.954844,8.078240,238.456864,42.014373,-22.108384,...,-0.000065,-0.000253,-0.000040,-0.000023,-0.000063,0.0,0.0,0.0,0.201,0.200
2,2026-08-25 11:34:55.852,235.874385,-43.976989,77.515954,293.099997,-67.954845,8.078178,238.456864,42.014373,-22.108384,...,-0.000069,-0.000195,-0.000013,-0.000017,-0.000055,0.0,0.0,0.0,0.401,0.400
3,2026-08-25 11:34:56.056,235.874385,-43.976989,77.515954,293.100897,-67.954862,8.078139,238.456864,42.014373,-22.108384,...,-0.000036,-0.000191,-0.000006,-0.000015,-0.000050,0.0,0.0,0.0,0.604,0.604
4,2026-08-25 11:34:56.297,235.874385,-43.976989,77.515954,293.101884,-67.954855,8.078171,238.456864,42.014373,-22.108384,...,0.000055,0.000049,0.000020,0.000003,-0.000014,0.0,0.0,0.0,0.846,0.845


# PID: Position Setpoint vs Present Value (θ, Base-frame motor angles)

In [4]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- θ_sp vs θ_pv (deg)" for ax in AXES], vertical_spacing=0.08)
for i, ax in enumerate(AXES):
    row = i + 1
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=pid_df[f"θ_sp_{row}"], name=f"{ax} θ_sp",
        line=dict(color="mediumseagreen", width=1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=pid_df[f"θ_pv_{row}"], name=f"{ax} θ_pv",
        line=dict(color="white", width=1)), row=row, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=800, width=1300, template="plotly_dark", hovermode="x unified",
    title="PID Setpoint vs Present Value", legend=dict(groupclick="toggleitem"))
fig.show()


# PID: Velocity breakdown (Kp / Ki / Kd / FF / Output)

Matches the Alpaca Pilot PID Tuning page convention: Cyan = Output (ω_op, what actually
drove the motor), Magenta = Kp, Olive = Ki, Orange = Kd, Green = FF (currently ω_ff − ω_pec
combined -- PEC is not yet broken out as its own field in the payload).

In [5]:
ARCSEC = 3600  # deg -> arcsec, for readability at the scale this investigation cares about

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- velocity components (arcsec/s)" for ax in AXES], vertical_spacing=0.08)
components = [("ω_ff", "green"), ("ω_kp", "magenta"), ("ω_ki", "olive"), ("ω_kd", "orange"), ("ω_op", "cyan")]
for i, ax in enumerate(AXES):
    row = i + 1
    for key, color in components:
        fig.add_trace(go.Scatter(x=pid_df.t_sec, y=pid_df[f"{key}_{row}"] * ARCSEC, name=f"{ax} {key}",
            line=dict(color=color, width=1)), row=row, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=900, width=1300, template="plotly_dark", hovermode="x unified",
    title="PID Velocity Breakdown", legend=dict(groupclick="toggleitem"))
fig.show()


# KF: Measured vs Filtered position/velocity state

In [6]:
fig = make_subplots(rows=3, cols=2, shared_xaxes=True,
    subplot_titles=sum([[f"{ax} -- θ_meas vs θ_state (deg)", f"{ax} -- ω_meas vs ω_state (arcsec/s)"] for ax in AXES], []),
    vertical_spacing=0.06)
for i, ax in enumerate(AXES):
    row = i + 1
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=kf_df[f"θ_meas_{row}"], name=f"{ax} θ_meas",
        line=dict(color="royalblue", width=1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=kf_df[f"θ_state_{row}"], name=f"{ax} θ_state",
        line=dict(color="white", width=1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=kf_df[f"ω_meas_{row}"] * ARCSEC, name=f"{ax} ω_meas",
        line=dict(color="royalblue", width=1)), row=row, col=2)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=kf_df[f"ω_state_{row}"] * ARCSEC, name=f"{ax} ω_state",
        line=dict(color="white", width=1)), row=row, col=2)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=2)
fig.update_layout(height=900, width=1500, template="plotly_dark", hovermode="x unified",
    title="Kalman Filter -- Measured vs Filtered State", legend=dict(groupclick="toggleitem"))
fig.show()


# KF: Gain per axis (position + velocity)

`K_gain` is the diagonal of the Kalman gain matrix: indices 1-3 are the position gains
(θ1-3), 4-6 the velocity gains (ω1-3). A gain spike means the filter suddenly started
trusting a raw measurement much more than usual -- worth cross-checking against any
θ_meas/θ_state divergence at the same tick.

In [7]:
fig = go.Figure()
labels = [f"{ax} pos" for ax in AXES] + [f"{ax} vel" for ax in AXES]
colors = ["royalblue", "orange", "mediumseagreen", "royalblue", "orange", "mediumseagreen"]
dashes = ["solid", "solid", "solid", "dash", "dash", "dash"]
for i, (label, color, dash) in enumerate(zip(labels, colors, dashes)):
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=kf_df[f"K_gain_{i+1}"], name=label,
        line=dict(color=color, width=1, dash=dash)))
fig.update_layout(height=500, width=1300, template="plotly_dark", hovermode="x unified",
    title="Kalman Gain per axis", xaxis_title="Time (s)", yaxis_title="Gain",
    legend=dict(groupclick="toggleitem"))
fig.show()


# Anomaly hunt: sudden PID reactions

Flags control ticks where `|ω_kp|` on any axis spikes far above its local baseline --
a direct proxy for "the PID suddenly thought it was a long way off target and kicked the
motor hard for one or two ticks", the kind of transient that could put a sharp bend in a
star trail without ever crossing the log's `WARNING` lag thresholds. Uses a rolling
median + MAD (robust to the spikes themselves, unlike mean/stdev).

In [8]:
WINDOW = 51       # ticks (~10s at 200ms) for the rolling baseline
N_MAD  = 8.0      # flag threshold, in robust standard deviations

anomalies = []
for i, ax in enumerate(AXES):
    col = f"ω_kp_{i+1}"
    series = (pid_df[col] * ARCSEC).abs()
    med = series.rolling(WINDOW, center=True, min_periods=WINDOW//2).median()
    mad = (series - med).abs().rolling(WINDOW, center=True, min_periods=WINDOW//2).median()
    robust_std = 1.4826 * mad
    threshold = med + N_MAD * robust_std
    flagged = pid_df[(series > threshold) & (robust_std > 1e-6)]
    for _, row in flagged.iterrows():
        anomalies.append(dict(axis=ax, t_sec=row.t_sec, timestamp=row.timestamp,
                               omega_kp_arcsec_s=row[col] * ARCSEC))

anomalies_df = pd.DataFrame(anomalies).sort_values("t_sec") if anomalies else pd.DataFrame()
print(f"Flagged {len(anomalies_df)} anomalous ticks across {len(AXES)} axes")
anomalies_df.head(30)


Flagged 0 anomalous ticks across 3 axes


""


In [9]:
# Zoom into context (±5s) around the single largest flagged anomaly, across every PID/KF component.
if len(anomalies_df):
    worst = anomalies_df.loc[anomalies_df.omega_kp_arcsec_s.idxmax()]
    print(f"Largest anomaly: axis={worst.axis}  t={worst.t_sec:.2f}s  |ω_kp|={worst.omega_kp_arcsec_s:.3f} arcsec/s")
    window = merged[(merged.t_sec > worst.t_sec - 5) & (merged.t_sec < worst.t_sec + 5)] if len(merged) else pd.DataFrame()
    display_cols = [c for c in window.columns if c.startswith(("θ_", "ω_", "t_sec"))]
    window[display_cols]
else:
    print("No anomalies flagged at this threshold -- try lowering N_MAD.")


No anomalies flagged at this threshold -- try lowering N_MAD.


# Notes